# Model Comparison Visualization

Compare v9, v12, v13, and refinement model outputs across val volumes.
Shows cross-sections where different comp score components succeed or fail.

**Models:**
- **v9** (traced): current best baseline, SegResNet, fit_one_cycle 30ep
- **v12** (ep20): flat_cos 50ep, same architecture
- **v13** (ep15): 3-class formulation, flat_cos 50ep
- **Refinement Phase 2**: learned post-processing on v9 probmaps

**Metrics:**
- TopoScore (30%): topology preservation — skeleton matching
- SurfaceDice@2 (35%): surface accuracy within 2-voxel tolerance
- VOI (35%): variation of information — segmentation quality

In [1]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import tifffile
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from pathlib import Path
from monai.networks.nets import SegResNet
from scipy.ndimage import (
    generate_binary_structure, binary_propagation,
    binary_closing, label as scipy_label, zoom as scipy_zoom,
    distance_transform_edt,
)
from topometrics.leaderboard import compute_leaderboard_score
import warnings
warnings.filterwarnings('ignore')

ROOT = Path('/workspace/vesuvius-kaggle-competition')
CKPT_DIR = ROOT / 'checkpoints' / 'models'
TRAIN_IMG = ROOT / 'data' / 'train_images'
TRAIN_LBL = ROOT / 'data' / 'train_labels'
PROBMAP_DIR = ROOT / 'data' / 'refinement_data' / 'probmaps'

PATCH_SIZE = 160
STRIDE = 80
T_LOW = 0.35
T_HIGH = 0.75
METRIC_DOWNSAMPLE = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

<frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


Device: cuda


## 1. Load models

In [2]:
def load_segresnet(checkpoint_path, out_channels=1):
    model = SegResNet(
        spatial_dims=3, in_channels=1, out_channels=out_channels,
        init_filters=16, blocks_down=[1, 2, 2, 4], blocks_up=[1, 1, 1],
        dropout_prob=0.2,
    )
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    state = ckpt.get('model', ckpt)
    model.load_state_dict(state)
    return model.to(device).eval()

# Refinement UNet (must match architecture in scripts/eval_refinement.py)
class ConvBlock3D(torch.nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = torch.nn.Sequential(
            torch.nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            torch.nn.GroupNorm(min(8, out_ch), out_ch),
            torch.nn.ReLU(inplace=True),
            torch.nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            torch.nn.GroupNorm(min(8, out_ch), out_ch),
            torch.nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.conv(x)

class RefinementUNet3D(torch.nn.Module):
    def __init__(self, in_ch=1, out_ch=1, channels=(8, 16, 32, 64), dropout=0.2):
        super().__init__()
        self.enc1 = ConvBlock3D(in_ch, channels[0])
        self.enc2 = ConvBlock3D(channels[0], channels[1])
        self.enc3 = ConvBlock3D(channels[1], channels[2])
        self.bottleneck = ConvBlock3D(channels[2], channels[3])
        self.dropout = torch.nn.Dropout3d(p=dropout)
        self.dec3 = ConvBlock3D(channels[3] + channels[2], channels[2])
        self.dec2 = ConvBlock3D(channels[2] + channels[1], channels[1])
        self.dec1 = ConvBlock3D(channels[1] + channels[0], channels[0])
        self.conv_final = torch.nn.Conv3d(channels[0], out_ch, 1)
        self.pool = torch.nn.MaxPool3d(2)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        b = self.dropout(b)
        d3 = F.interpolate(b, size=e3.shape[2:], mode='trilinear', align_corners=False)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = F.interpolate(d3, size=e2.shape[2:], mode='trilinear', align_corners=False)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = F.interpolate(d2, size=e1.shape[2:], mode='trilinear', align_corners=False)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return self.conv_final(d1)

# Load all models
print('Loading models...')
model_v9 = torch.jit.load(str(ROOT / 'kaggle' / 'kaggle_weights_download' / 'best_segresnet_v9_traced.pt'), map_location=device)
model_v9.eval()
print('  v9 (traced) loaded')

model_v12 = load_segresnet(CKPT_DIR / 'segresnet_v12_ep20.pth', out_channels=1)
print('  v12 ep20 loaded')

model_v13 = load_segresnet(CKPT_DIR / 'segresnet_v13_ep15.pth', out_channels=3)
print('  v13 ep15 loaded (3-class)')

model_ref = RefinementUNet3D()
ref_state = torch.load(CKPT_DIR / 'best_refinement_phase2.pth', map_location='cpu', weights_only=False)
model_ref.load_state_dict(ref_state)
model_ref = model_ref.to(device).eval()
print('  Refinement Phase 2 loaded')

print('All models loaded.')

Loading models...


  v9 (traced) loaded

  v12 ep20 loaded


  v13 ep15 loaded (3-class)
  Refinement Phase 2 loaded
All models loaded.


## 2. Inference utilities

In [3]:
def build_gaussian_map(ps=PATCH_SIZE, sigma_scale=0.125):
    sigma = ps * sigma_scale
    ax = np.arange(ps, dtype=np.float32) - (ps - 1) / 2
    g1d = np.exp(-0.5 * (ax / sigma) ** 2)
    g3d = g1d[:, None, None] * g1d[None, :, None] * g1d[None, None, :]
    return g3d / g3d.max()

GAUSSIAN_MAP = build_gaussian_map()

def _positions(length, ps, stride):
    pos = list(range(0, length - ps, stride)) + [length - ps]
    return sorted(set(pos))

def swi_gaussian(model, volume, out_channels=1):
    """Gaussian SWI — returns raw logits (C, D, H, W)."""
    D, H, W = volume.shape
    ps = PATCH_SIZE
    gauss = GAUSSIAN_MAP
    output = np.zeros((out_channels, D, H, W), dtype=np.float32)
    weights = np.zeros((D, H, W), dtype=np.float32)
    z_pos = _positions(D, ps, STRIDE)
    y_pos = _positions(H, ps, STRIDE)
    x_pos = _positions(W, ps, STRIDE)
    with torch.no_grad(), torch.amp.autocast('cuda'):
        for z in z_pos:
            for y in y_pos:
                for x in x_pos:
                    patch = volume[z:z+ps, y:y+ps, x:x+ps]
                    patch_t = torch.from_numpy(patch).float().unsqueeze(0).unsqueeze(0).to(device) / 255.0
                    logits = model(patch_t).squeeze(0).cpu().numpy()
                    if out_channels == 1:
                        logits = logits.squeeze(0)
                        output[0, z:z+ps, y:y+ps, x:x+ps] += logits * gauss
                    else:
                        for c in range(out_channels):
                            output[c, z:z+ps, y:y+ps, x:x+ps] += logits[c] * gauss
                    weights[z:z+ps, y:y+ps, x:x+ps] += gauss
    for c in range(out_channels):
        output[c] /= weights
    return output

def logits_to_prob(logits, three_class=False):
    if three_class:
        logits_t = torch.from_numpy(logits).unsqueeze(0)
        probs = F.softmax(logits_t, dim=1).squeeze(0).numpy()
        return probs[1]
    else:
        return 1.0 / (1.0 + np.exp(-logits[0]))

def hysteresis_threshold(prob, t_low=T_LOW, t_high=T_HIGH):
    strong = prob >= t_high
    weak = prob >= t_low
    struct = generate_binary_structure(3, 3)
    return binary_propagation(strong, structure=struct, mask=weak).astype(np.uint8)

def postprocess(prob):
    binary = hysteresis_threshold(prob)
    struct = generate_binary_structure(3, 3)
    z_struct = np.zeros((3, 3, 3), dtype=bool)
    z_struct[1] = struct[1]  # anisotropic: wider in-plane
    closed = binary_closing(binary, structure=z_struct, iterations=2).astype(np.uint8)
    lbl_arr, n_comp = scipy_label(closed)
    if n_comp > 1:
        sizes = np.bincount(lbl_arr.ravel())[1:]
        keep = np.argmax(sizes) + 1
        closed = (lbl_arr == keep).astype(np.uint8)
    return closed

def refinement_predict(probmap, model):
    """Run refinement model on a probmap."""
    with torch.no_grad():
        inp = torch.from_numpy(probmap.astype(np.float32)).unsqueeze(0).unsqueeze(0).to(device)
        logits = model(inp)
        return torch.sigmoid(logits).squeeze().cpu().numpy()

def score(pred, lbl, ds=METRIC_DOWNSAMPLE):
    scale = 1.0 / ds
    pred_ds = scipy_zoom(pred, scale, order=0).astype(np.uint8)
    lbl_ds = scipy_zoom(lbl, scale, order=0).astype(np.uint8)
    return compute_leaderboard_score(
        pred_ds, lbl_ds, ignore_label=2, spacing=(1,1,1),
        surface_tolerance=2.0, voi_alpha=0.3, combine_weights=(0.3, 0.35, 0.35),
    )

print('Utilities ready.')

Utilities ready.


## 3. Select volumes and run inference

Pick a few val volumes with varying performance characteristics.

In [4]:
import time

train_df = pd.read_csv(ROOT / 'data' / 'train.csv')
val_df = train_df[train_df.scroll_id == 26002]

# Pick 5 volumes for detailed visualization
# Use the first 5 that have probmaps (for refinement comparison)
probmap_ids = set(int(p.stem) for p in PROBMAP_DIR.glob('*.npy'))
val_ids = [vid for vid in val_df.id.tolist() if vid in probmap_ids][:5]
print(f'Visualizing {len(val_ids)} volumes: {val_ids}')

results = []
volume_data = {}  # Store predictions for visualization

for i, vid in enumerate(val_ids):
    print(f'\n[{i+1}/{len(val_ids)}] Volume {vid}')
    t0 = time.time()
    
    # Load data
    img = tifffile.imread(TRAIN_IMG / f'{vid}.tif')
    lbl = tifffile.imread(TRAIN_LBL / f'{vid}.tif')
    v9_prob = np.load(PROBMAP_DIR / f'{vid}.npy')
    
    # v9: use pre-computed probmap (these were generated with full pipeline)
    pred_v9 = postprocess(v9_prob)
    
    # v12: run SWI
    logits_v12 = swi_gaussian(model_v12, img, out_channels=1)
    prob_v12 = logits_to_prob(logits_v12, three_class=False)
    pred_v12 = postprocess(prob_v12)
    
    # v13: run SWI (3-class)
    logits_v13 = swi_gaussian(model_v13, img, out_channels=3)
    prob_v13 = logits_to_prob(logits_v13, three_class=True)
    pred_v13 = postprocess(prob_v13)
    
    # Refinement: use v9 probmap as input
    ref_prob = refinement_predict(v9_prob, model_ref)
    pred_ref_raw = (ref_prob > 0.5).astype(np.uint8)
    pred_ref_pp = postprocess(ref_prob)  # Option A
    
    # Score all
    r_v9 = score(pred_v9, lbl)
    r_v12 = score(pred_v12, lbl)
    r_v13 = score(pred_v13, lbl)
    r_ref = score(pred_ref_raw, lbl)
    r_refpp = score(pred_ref_pp, lbl)
    
    elapsed = time.time() - t0
    print(f'  v9={r_v9.score:.4f}  v12={r_v12.score:.4f}  v13={r_v13.score:.4f}  '
          f'ref={r_ref.score:.4f}  ref+pp={r_refpp.score:.4f}  ({elapsed:.0f}s)')
    
    results.append({
        'vid': vid,
        'v9': r_v9.score, 'v9_topo': r_v9.topo.toposcore, 'v9_sdice': r_v9.surface_dice, 'v9_voi': r_v9.voi.voi_score,
        'v12': r_v12.score, 'v12_topo': r_v12.topo.toposcore, 'v12_sdice': r_v12.surface_dice, 'v12_voi': r_v12.voi.voi_score,
        'v13': r_v13.score, 'v13_topo': r_v13.topo.toposcore, 'v13_sdice': r_v13.surface_dice, 'v13_voi': r_v13.voi.voi_score,
        'ref': r_ref.score, 'ref_topo': r_ref.topo.toposcore, 'ref_sdice': r_ref.surface_dice, 'ref_voi': r_ref.voi.voi_score,
        'ref_pp': r_refpp.score, 'ref_pp_topo': r_refpp.topo.toposcore, 'ref_pp_sdice': r_refpp.surface_dice, 'ref_pp_voi': r_refpp.voi.voi_score,
    })
    
    volume_data[vid] = {
        'img': img, 'lbl': lbl,
        'prob_v9': v9_prob, 'prob_v12': prob_v12, 'prob_v13': prob_v13, 'prob_ref': ref_prob,
        'pred_v9': pred_v9, 'pred_v12': pred_v12, 'pred_v13': pred_v13,
        'pred_ref': pred_ref_raw, 'pred_ref_pp': pred_ref_pp,
    }

df = pd.DataFrame(results)
print('\nDone.')

Visualizing 5 volumes: [26894125, 105796630, 327851248, 418613908, 477109023]

[1/5] Volume 26894125


  v9=0.4920  v12=0.4947  v13=0.4858  ref=0.4728  ref+pp=0.5162  (39s)

[2/5] Volume 105796630


  v9=0.4877  v12=0.4941  v13=0.4786  ref=0.4679  ref+pp=0.4943  (35s)

[3/5] Volume 327851248


  v9=0.5028  v12=0.5032  v13=0.5019  ref=0.4895  ref+pp=0.5033  (86s)

[4/5] Volume 418613908


  v9=0.5165  v12=0.5159  v13=0.5122  ref=0.4736  ref+pp=0.5173  (39s)

[5/5] Volume 477109023


  v9=0.5172  v12=0.5258  v13=0.5303  ref=0.4399  ref+pp=0.5208  (22s)

Done.


## 4. Score summary table

In [5]:
print('CompScore by volume:')
print(df[['vid', 'v9', 'v12', 'v13', 'ref', 'ref_pp']].to_string(index=False, float_format='%.4f'))
print()
print('Mean CompScore:')
for col in ['v9', 'v12', 'v13', 'ref', 'ref_pp']:
    print(f'  {col:>8s}: {df[col].mean():.4f}')

print('\nComponent breakdown (mean):')
print(f"  {'':>8s}  {'Topo':>8s}  {'SDice':>8s}  {'VOI':>8s}  {'Comp':>8s}")
for name in ['v9', 'v12', 'v13', 'ref', 'ref_pp']:
    print(f"  {name:>8s}  {df[f'{name}_topo'].mean():8.4f}  {df[f'{name}_sdice'].mean():8.4f}  "
          f"{df[f'{name}_voi'].mean():8.4f}  {df[name].mean():8.4f}")

CompScore by volume:
      vid     v9    v12    v13    ref  ref_pp
 26894125 0.4920 0.4947 0.4858 0.4728  0.5162
105796630 0.4877 0.4941 0.4786 0.4679  0.4943
327851248 0.5028 0.5032 0.5019 0.4895  0.5033
418613908 0.5165 0.5159 0.5122 0.4736  0.5173
477109023 0.5172 0.5258 0.5303 0.4399  0.5208

Mean CompScore:
        v9: 0.5032
       v12: 0.5067
       v13: 0.5018
       ref: 0.4688
    ref_pp: 0.5104

Component breakdown (mean):
                Topo     SDice       VOI      Comp
        v9    0.0106    0.7117    0.7169    0.5032
       v12    0.0218    0.7190    0.7101    0.5067
       v13    0.0156    0.6970    0.7232    0.5018
       ref    0.0516    0.7402    0.5550    0.4688
    ref_pp    0.0432    0.7493    0.6719    0.5104


## 5. Cross-section visualization: predictions vs ground truth

For each volume, show center slices along all 3 axes with:
- CT image
- Ground truth label
- Each model's binary prediction
- Overlay showing TP (green), FP (red), FN (blue)

In [6]:
def make_overlay(pred, lbl):
    """Create TP/FP/FN overlay. Returns RGB image."""
    mask = (lbl != 2)  # ignore unlabeled
    gt = (lbl == 1)
    tp = pred & gt & mask  # green
    fp = pred & ~gt & mask  # red
    fn = ~pred & gt & mask  # blue
    rgb = np.zeros((*pred.shape, 3), dtype=np.float32)
    rgb[tp] = [0, 1, 0]
    rgb[fp] = [1, 0, 0]
    rgb[fn] = [0, 0.4, 1]
    return rgb

def plot_cross_sections(vid, data, axis=0, slice_idx=None):
    """Plot center cross-section along given axis for all models."""
    img = data['img']
    lbl = data['lbl']
    
    if slice_idx is None:
        # Find slice with most foreground
        fg_counts = (lbl == 1).sum(axis=tuple(i for i in range(3) if i != axis))
        slice_idx = int(np.argmax(fg_counts))
    
    axis_names = ['Z (axial)', 'Y (coronal)', 'X (sagittal)']
    
    def take_slice(arr, ax, idx):
        slc = [slice(None)] * 3
        slc[ax] = idx
        return arr[tuple(slc)]
    
    img_s = take_slice(img, axis, slice_idx)
    lbl_s = take_slice(lbl, axis, slice_idx)
    
    models = [
        ('v9', 'pred_v9'),
        ('v12', 'pred_v12'),
        ('v13', 'pred_v13'),
        ('Ref (raw)', 'pred_ref'),
        ('Ref+PP', 'pred_ref_pp'),
    ]
    
    fig, axes = plt.subplots(2, len(models) + 2, figsize=(4 * (len(models) + 2), 8))
    fig.suptitle(f'Volume {vid} — {axis_names[axis]} slice {slice_idx}', fontsize=14)
    
    # Row 1: CT image, GT, then each model's prediction
    axes[0, 0].imshow(img_s, cmap='gray')
    axes[0, 0].set_title('CT Image')
    axes[0, 0].axis('off')
    
    # GT with label colors
    gt_vis = np.zeros((*lbl_s.shape, 3), dtype=np.float32)
    gt_vis[lbl_s == 1] = [0, 1, 0]  # fg = green
    gt_vis[lbl_s == 2] = [0.5, 0.5, 0.5]  # unlabeled = gray
    axes[0, 1].imshow(gt_vis)
    axes[0, 1].set_title('Ground Truth')
    axes[0, 1].axis('off')
    
    for j, (name, key) in enumerate(models):
        pred_s = take_slice(data[key], axis, slice_idx).astype(bool)
        axes[0, j+2].imshow(pred_s, cmap='gray')
        axes[0, j+2].set_title(name)
        axes[0, j+2].axis('off')
    
    # Row 2: empty, empty, then overlays (TP/FP/FN)
    # Probability maps for first two
    prob_keys = [('prob_v9', 'v9 prob'), ('prob_v12', 'v12 prob')]
    for j, (pk, pname) in enumerate(prob_keys):
        prob_s = take_slice(data[pk], axis, slice_idx)
        axes[1, j].imshow(prob_s, cmap='hot', vmin=0, vmax=1)
        axes[1, j].set_title(pname)
        axes[1, j].axis('off')
    
    for j, (name, key) in enumerate(models):
        pred_s = take_slice(data[key], axis, slice_idx).astype(bool)
        overlay = make_overlay(pred_s, lbl_s)
        axes[1, j+2].imshow(overlay)
        axes[1, j+2].set_title(f'{name} (G=TP R=FP B=FN)')
        axes[1, j+2].axis('off')
    
    plt.tight_layout()
    plt.show()

print('Visualization functions ready.')

Visualization functions ready.


## 6. Cross-section views for each volume

Shows the slice with most foreground along Z axis.

In [7]:
for vid in val_ids:
    data = volume_data[vid]
    r = df[df.vid == vid].iloc[0]
    print(f'\nVolume {vid}:')
    print(f'  CompScore:  v9={r.v9:.4f}  v12={r.v12:.4f}  v13={r.v13:.4f}  ref={r.ref:.4f}  ref+pp={r.ref_pp:.4f}')
    print(f'  TopoScore:  v9={r.v9_topo:.4f}  v12={r.v12_topo:.4f}  v13={r.v13_topo:.4f}  ref={r.ref_topo:.4f}  ref+pp={r.ref_pp_topo:.4f}')
    print(f'  SurfDice:   v9={r.v9_sdice:.4f}  v12={r.v12_sdice:.4f}  v13={r.v13_sdice:.4f}  ref={r.ref_sdice:.4f}  ref+pp={r.ref_pp_sdice:.4f}')
    print(f'  VOI:        v9={r.v9_voi:.4f}  v12={r.v12_voi:.4f}  v13={r.v13_voi:.4f}  ref={r.ref_voi:.4f}  ref+pp={r.ref_pp_voi:.4f}')
    
    # Z-axis (most informative for surface detection)
    plot_cross_sections(vid, data, axis=0)


Volume 26894125:
  CompScore:  v9=0.4920  v12=0.4947  v13=0.4858  ref=0.4728  ref+pp=0.5162
  TopoScore:  v9=0.0047  v12=0.0068  v13=0.0055  ref=0.0205  ref+pp=0.0514
  SurfDice:   v9=0.6716  v12=0.6777  v13=0.6481  ref=0.7542  ref+pp=0.7550
  VOI:        v9=0.7300  v12=0.7298  v13=0.7351  ref=0.5790  ref+pp=0.6758

Volume 105796630:
  CompScore:  v9=0.4877  v12=0.4941  v13=0.4786  ref=0.4679  ref+pp=0.4943
  TopoScore:  v9=0.0157  v12=0.0324  v13=0.0102  ref=0.0698  ref+pp=0.0416
  SurfDice:   v9=0.6749  v12=0.6826  v13=0.6401  ref=0.7391  ref+pp=0.7259
  VOI:        v9=0.7049  v12=0.7013  v13=0.7186  ref=0.5381  ref+pp=0.6508



Volume 327851248:
  CompScore:  v9=0.5028  v12=0.5032  v13=0.5019  ref=0.4895  ref+pp=0.5033
  TopoScore:  v9=0.0079  v12=0.0119  v13=0.0090  ref=0.1259  ref+pp=0.0408
  SurfDice:   v9=0.7113  v12=0.7253  v13=0.7050  ref=0.7338  ref+pp=0.7431
  VOI:        v9=0.7186  v12=0.7023  v13=0.7212  ref=0.5569  ref+pp=0.6601

Volume 418613908:
  CompScore:  v9=0.5165  v12=0.5159  v13=0.5122  ref=0.4736  ref+pp=0.5173
  TopoScore:  v9=0.0118  v12=0.0155  v13=0.0119  ref=0.0201  ref+pp=0.0385
  SurfDice:   v9=0.7737  v12=0.7745  v13=0.7510  ref=0.7636  ref+pp=0.7870
  VOI:        v9=0.6917  v12=0.6861  v13=0.7022  ref=0.5724  ref+pp=0.6580



Volume 477109023:
  CompScore:  v9=0.5172  v12=0.5258  v13=0.5303  ref=0.4399  ref+pp=0.5208
  TopoScore:  v9=0.0131  v12=0.0424  v13=0.0414  ref=0.0215  ref+pp=0.0439
  SurfDice:   v9=0.7269  v12=0.7351  v13=0.7408  ref=0.7101  ref+pp=0.7355
  VOI:        v9=0.7395  v12=0.7310  v13=0.7390  ref=0.5284  ref+pp=0.7148


## 7. Probability map comparison

Compare probability distributions across models to understand confidence and calibration.

In [8]:
fig, axes = plt.subplots(len(val_ids), 4, figsize=(16, 4*len(val_ids)))
if len(val_ids) == 1:
    axes = axes.reshape(1, -1)

for i, vid in enumerate(val_ids):
    data = volume_data[vid]
    lbl = data['lbl']
    fg_mask = (lbl == 1)
    bg_mask = (lbl == 0)
    
    for j, (prob_key, name) in enumerate([
        ('prob_v9', 'v9'), ('prob_v12', 'v12'), ('prob_v13', 'v13'), ('prob_ref', 'Refinement')
    ]):
        prob = data[prob_key]
        ax = axes[i, j]
        
        # Histogram of probs in FG vs BG regions
        fg_probs = prob[fg_mask]
        bg_probs = prob[bg_mask]
        
        ax.hist(bg_probs.ravel(), bins=50, alpha=0.5, label='BG', color='blue', density=True)
        ax.hist(fg_probs.ravel(), bins=50, alpha=0.5, label='FG', color='green', density=True)
        ax.axvline(T_HIGH, color='red', ls='--', label=f'T_HIGH={T_HIGH}')
        ax.axvline(T_LOW, color='orange', ls='--', label=f'T_LOW={T_LOW}')
        ax.set_title(f'{name} — vol {vid}')
        ax.set_xlabel('Probability')
        ax.set_xlim(0, 1)
        if j == 0:
            ax.set_ylabel('Density')
        ax.legend(fontsize=7)

plt.suptitle('Probability distributions: FG (green) vs BG (blue) regions', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Connected component analysis

VOI cares about connected component quality. Compare fragment counts.

In [9]:
print(f'{"Volume":>12s}  {"GT":>4s}  {"v9":>4s}  {"v12":>4s}  {"v13":>4s}  {"Ref":>4s}  {"Ref+PP":>6s}')
print('-' * 55)

for vid in val_ids:
    data = volume_data[vid]
    lbl = data['lbl']
    gt_binary = (lbl == 1).astype(np.uint8)
    
    _, n_gt = scipy_label(gt_binary)
    _, n_v9 = scipy_label(data['pred_v9'])
    _, n_v12 = scipy_label(data['pred_v12'])
    _, n_v13 = scipy_label(data['pred_v13'])
    _, n_ref = scipy_label(data['pred_ref'])
    _, n_refpp = scipy_label(data['pred_ref_pp'])
    
    print(f'{vid:>12d}  {n_gt:>4d}  {n_v9:>4d}  {n_v12:>4d}  {n_v13:>4d}  {n_ref:>4d}  {n_refpp:>6d}')

print('\nFewer components = better for VOI (ideally matches GT).')
print('Refinement raw typically has many fragments; Ref+PP (closing+dust) reduces them.')

      Volume    GT    v9   v12   v13   Ref  Ref+PP
-------------------------------------------------------


    26894125     5     1     1     1  1231       1


   105796630     9     1     1     1   612       1


   327851248     7     1     1     1  2481       1


   418613908     6     1     1     1  1794       1


   477109023     4     1     1     1  1457       1

Fewer components = better for VOI (ideally matches GT).
Refinement raw typically has many fragments; Ref+PP (closing+dust) reduces them.


## 9. Metric correlation heatmap

How do the three metrics correlate with each other across volumes?

In [10]:
# Collect all metric values across all models and volumes
metric_data = []
for _, row in df.iterrows():
    for model in ['v9', 'v12', 'v13', 'ref', 'ref_pp']:
        metric_data.append({
            'model': model,
            'vid': row['vid'],
            'TopoScore': row[f'{model}_topo'],
            'SurfaceDice': row[f'{model}_sdice'],
            'VOI': row[f'{model}_voi'],
            'CompScore': row[model],
        })

mdf = pd.DataFrame(metric_data)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

pairs = [('TopoScore', 'VOI'), ('TopoScore', 'SurfaceDice'), ('SurfaceDice', 'VOI')]
colors = {'v9': 'blue', 'v12': 'green', 'v13': 'orange', 'ref': 'red', 'ref_pp': 'purple'}

for ax, (x_col, y_col) in zip(axes, pairs):
    for model, color in colors.items():
        subset = mdf[mdf.model == model]
        ax.scatter(subset[x_col], subset[y_col], c=color, label=model, alpha=0.7, s=50)
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.legend(fontsize=8)
    ax.set_title(f'{x_col} vs {y_col}')

plt.suptitle('Metric relationships across models and volumes', fontsize=14)
plt.tight_layout()
plt.show()

## 10. Surface error distance maps

Where exactly is each model making surface errors? Compute distance from prediction surface to GT surface.

In [11]:
def surface_distance_map(pred, gt_binary):
    """Compute per-voxel distance from predicted surface to GT surface."""
    # Get surfaces (boundary voxels)
    from scipy.ndimage import binary_erosion
    struct = generate_binary_structure(3, 1)
    pred_surf = pred.astype(bool) & ~binary_erosion(pred.astype(bool), struct)
    gt_surf = gt_binary.astype(bool) & ~binary_erosion(gt_binary.astype(bool), struct)
    
    # Distance from pred surface to nearest GT surface
    if gt_surf.any():
        gt_dist = distance_transform_edt(~gt_surf)
        return gt_dist * pred_surf  # distance only at predicted surface voxels
    return np.zeros_like(pred, dtype=np.float32)

# Visualize for first volume
vid = val_ids[0]
data = volume_data[vid]
lbl = data['lbl']
gt_binary = (lbl == 1).astype(np.uint8)

# Find best Z slice
fg_counts = (lbl == 1).sum(axis=(1, 2))
z_slice = int(np.argmax(fg_counts))

fig, axes = plt.subplots(1, 5, figsize=(25, 5))
fig.suptitle(f'Surface error distance — Volume {vid}, Z={z_slice}', fontsize=14)

for j, (name, key) in enumerate([
    ('v9', 'pred_v9'), ('v12', 'pred_v12'), ('v13', 'pred_v13'),
    ('Ref', 'pred_ref'), ('Ref+PP', 'pred_ref_pp')
]):
    dist_map = surface_distance_map(data[key], gt_binary)
    im = axes[j].imshow(dist_map[z_slice], cmap='hot', vmin=0, vmax=10)
    axes[j].set_title(f'{name} (max err={dist_map[z_slice].max():.1f})')
    axes[j].axis('off')

plt.colorbar(im, ax=axes[-1], shrink=0.8, label='Distance to GT surface')
plt.tight_layout()
plt.show()

## 11. Y-Z cross sections (sagittal view)

These show the surface topology more clearly — fragmentation and gaps are visible.

In [12]:
for vid in val_ids:
    data = volume_data[vid]
    r = df[df.vid == vid].iloc[0]
    print(f'\nVolume {vid} — Sagittal (X-axis) view:')
    print(f'  VOI:  v9={r.v9_voi:.4f}  v12={r.v12_voi:.4f}  ref={r.ref_voi:.4f}  ref+pp={r.ref_pp_voi:.4f}')
    plot_cross_sections(vid, data, axis=2)  # X-axis = sagittal


Volume 26894125 — Sagittal (X-axis) view:
  VOI:  v9=0.7300  v12=0.7298  ref=0.5790  ref+pp=0.6758

Volume 105796630 — Sagittal (X-axis) view:
  VOI:  v9=0.7049  v12=0.7013  ref=0.5381  ref+pp=0.6508



Volume 327851248 — Sagittal (X-axis) view:
  VOI:  v9=0.7186  v12=0.7023  ref=0.5569  ref+pp=0.6601



Volume 418613908 — Sagittal (X-axis) view:
  VOI:  v9=0.6917  v12=0.6861  ref=0.5724  ref+pp=0.6580

Volume 477109023 — Sagittal (X-axis) view:
  VOI:  v9=0.7395  v12=0.7310  ref=0.5284  ref+pp=0.7148


## 12. Refinement model effect: before vs after

Side-by-side comparison of v9 probmap → refinement probmap → binary predictions.

In [13]:
for vid in val_ids:
    data = volume_data[vid]
    lbl = data['lbl']
    
    # Find best Z slice
    fg_counts = (lbl == 1).sum(axis=(1, 2))
    z = int(np.argmax(fg_counts))
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    fig.suptitle(f'Refinement effect — Volume {vid}, Z={z}', fontsize=14)
    
    # Row 1: probabilities
    axes[0, 0].imshow(data['prob_v9'][z], cmap='hot', vmin=0, vmax=1)
    axes[0, 0].set_title('v9 probmap (input)')
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(data['prob_ref'][z], cmap='hot', vmin=0, vmax=1)
    axes[0, 1].set_title('Refinement output')
    axes[0, 1].axis('off')
    
    # Difference map
    diff = data['prob_ref'][z] - data['prob_v9'][z]
    axes[0, 2].imshow(diff, cmap='RdBu_r', vmin=-0.5, vmax=0.5)
    axes[0, 2].set_title('Refinement - v9 (red=higher, blue=lower)')
    axes[0, 2].axis('off')
    
    # GT
    gt_vis = np.zeros((*lbl[z].shape, 3), dtype=np.float32)
    gt_vis[lbl[z] == 1] = [0, 1, 0]
    gt_vis[lbl[z] == 2] = [0.5, 0.5, 0.5]
    axes[0, 3].imshow(gt_vis)
    axes[0, 3].set_title('Ground Truth')
    axes[0, 3].axis('off')
    
    # Row 2: binary predictions with overlays
    for j, (name, key) in enumerate([
        ('v9 baseline', 'pred_v9'),
        ('Ref (raw)', 'pred_ref'),
        ('Ref+PP (Option A)', 'pred_ref_pp'),
    ]):
        pred_s = data[key][z].astype(bool)
        overlay = make_overlay(pred_s, lbl[z])
        axes[1, j].imshow(overlay)
        axes[1, j].set_title(f'{name} (G=TP R=FP B=FN)')
        axes[1, j].axis('off')
    
    axes[1, 3].imshow(data['img'][z], cmap='gray')
    axes[1, 3].set_title('CT Image')
    axes[1, 3].axis('off')
    
    plt.tight_layout()
    plt.show()

## 13. Per-volume metric breakdown bar chart

In [14]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics = ['topo', 'sdice', 'voi']
metric_names = ['TopoScore (30%)', 'SurfaceDice (35%)', 'VOI Score (35%)']
model_names = ['v9', 'v12', 'v13', 'ref', 'ref_pp']
model_colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0']

x = np.arange(len(val_ids))
width = 0.15

for ax, metric, mname in zip(axes, metrics, metric_names):
    for j, (model, color) in enumerate(zip(model_names, model_colors)):
        vals = [df[df.vid == vid][f'{model}_{metric}'].values[0] for vid in val_ids]
        ax.bar(x + j * width, vals, width, label=model, color=color, alpha=0.8)
    ax.set_xlabel('Volume')
    ax.set_ylabel(mname)
    ax.set_title(mname)
    ax.set_xticks(x + 2 * width)
    ax.set_xticklabels([str(v)[-4:] for v in val_ids], rotation=45)
    ax.legend(fontsize=8)

plt.suptitle('Per-volume metric breakdown by model', fontsize=14)
plt.tight_layout()
plt.show()